# B2-019-attention-transformers — Practice p15 — Solution

**Type:** proof · **Difficulty:** advanced · **Concepts:** multi-head-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

With \(X:(B,N,D)\) and \(W_Q,W_K,W_V:(D,D)\), the three projections are \((B,N,D)\). Set \(d_h=D/h\); reshaping to \((B,N,h,d_h)\) and transposing gives \((B,h,N,d_h)\). Per-head \(QK^\top\) produces \((B,h,N,N)\), softmax preserves it, and multiplying V gives \((B,h,N,d_h)\). Transpose back to \((B,N,h,d_h)\); flattening the last two axes restores width \(h d_h=D\), so concatenation is \((B,N,D)\). The output projection \((D,D)\) preserves that shape. Concatenating on sequence instead would create length \(hN\), mixing head identity with token identity and breaking residual alignment.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 0.0
RTOL = 0.0
B, N, D, h = 2, 3, 8, 2
d_h = D // h
X = np.arange(B * N * D, dtype=np.float64).reshape(B, N, D)
W_Q = W_K = W_V = W_O = np.eye(D, dtype=np.float64)
Q, K, V = X @ W_Q, X @ W_K, X @ W_V
Qh = Q.reshape(B, N, h, d_h).transpose(0, 2, 1, 3)
Kh = K.reshape(B, N, h, d_h).transpose(0, 2, 1, 3)
Vh = V.reshape(B, N, h, d_h).transpose(0, 2, 1, 3)
scores = Qh @ Kh.transpose(0, 1, 3, 2)
head_values = Vh + 100.0 * np.arange(h, dtype=np.float64)[None, :, None, None]
concatenated = head_values.transpose(0, 2, 1, 3).reshape(B, N, D)
projected = concatenated @ W_O
EXPECTED_CONCATENATED_B0_N0 = np.array([0.0, 1.0, 2.0, 3.0, 104.0, 105.0, 106.0, 107.0], dtype=np.float64)
EXPECTED_CONCATENATED_B1_N2 = np.array([40.0, 41.0, 42.0, 43.0, 144.0, 145.0, 146.0, 147.0], dtype=np.float64)

### Answer check

In [ ]:
assert Q.shape == K.shape == V.shape == (B, N, D)
assert Qh.shape == Kh.shape == Vh.shape == (B, h, N, d_h)
assert scores.shape == (B, h, N, N)
assert head_values.shape == (B, h, N, d_h)
assert concatenated.shape == projected.shape == (B, N, D)
np.testing.assert_allclose(concatenated[0, 0], EXPECTED_CONCATENATED_B0_N0, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(concatenated[1, 2], EXPECTED_CONCATENATED_B1_N2, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(projected, concatenated, atol=ATOL, rtol=RTOL)
assert h * d_h == D